## Challenge #1: Join to Range
<ins>Exercise #1 Join to Range:</ins>

>A company in Australia has source data which is made up of a series of postal codes (eg. 2000, 2001, 2002 etc.) amongst some other data fields. They have a separate reference table which contains postcode ranges (eg. 2000 to 2002) which they would like to use to match/filter their main data.
>
>Each Customer Record needs to be joined to the Lookup table based on a Postal Area Ranged region. Then finally summarize the customer data by Region, Sales Rep, and Responder, then a count of customers.
>
>Check and see what the result should look like by looking at the data labeled 'Output'.  Your mission is to take the input files and blend them so your result matches the output shown.  Good luck!

In [82]:
import pandas as pd

In [83]:
# preview output
output = pd.read_csv('output.csv', index_col = False)
df_output = pd.DataFrame(output)
df_output

,Region,Sales Rep,Responder,Count
0,R1,John,No,476
1,R1,John,Yes,76
2,R2,Ted,No,415
3,R2,Ted,Yes,87
4,R3,Nick,No,493
5,R3,Nick,Yes,92
6,R4,Mike,No,430
7,R4,Mike,Yes,82
8,R5,Paul,No,434
9,R5,Paul,Yes,93


In [84]:
# read and preview customers table
customer_records = pd.read_csv('input1.csv', index_col = False)
customer_records.head()

,Customer ID,Store Number,Customer Segment,Responder,Postal Area
0,2,100,Corporate,No,2086
1,3,100,Corporate,No,2051
2,5,100,Home Office,No,2077
3,6,106,Home Office,No,2004
4,8,101,Home Office,No,2010


In [85]:
# read and preview postal codes table
postal_codes = pd.read_csv('input2.csv', index_col = False)
postal_codes

,Range,Region,Sales Rep,Expect Revenue
0,2000-2019,R1,John,1000000
1,2020-2039,R2,Ted,3245234
2,2040-2059,R3,Nick,456654
3,2060-2079,R4,Mike,234545
4,2080-2100,R5,Paul,1232345


In [86]:
# create DataFrame
df_postal_codes = pd.DataFrame(postal_codes)
df_postal_codes

,Range,Region,Sales Rep,Expect Revenue
0,2000-2019,R1,John,1000000
1,2020-2039,R2,Ted,3245234
2,2040-2059,R3,Nick,456654
3,2060-2079,R4,Mike,234545
4,2080-2100,R5,Paul,1232345


In [87]:
# splits Range column into start and end values, casts strings to integers
df_postal_codes[['R1', 'R2']] = df_postal_codes['Range'].str.split('-', expand=True).astype("int32")
df_postal_codes

,Range,Region,Sales Rep,Expect Revenue,R1,R2
0,2000-2019,R1,John,1000000,2000,2019
1,2020-2039,R2,Ted,3245234,2020,2039
2,2040-2059,R3,Nick,456654,2040,2059
3,2060-2079,R4,Mike,234545,2060,2079
4,2080-2100,R5,Paul,1232345,2080,2100


In [88]:
# creates a column with a range between R1 and R2+1
df_postal_codes['Postal Area'] = df_postal_codes.apply(lambda x: range(x['R1'], x['R2'] + 1), axis=1)
# transform each element of range into a row, and drops range columns
df_postal_codes = df_postal_codes.explode('Postal Area').drop(['R1', 'R2'], axis=1).reset_index(drop=True)

In [89]:
df_postal_codes.head(5)

,Range,Region,Sales Rep,Expect Revenue,Postal Area
0,2000-2019,R1,John,1000000,2000
1,2000-2019,R1,John,1000000,2001
2,2000-2019,R1,John,1000000,2002
3,2000-2019,R1,John,1000000,2003
4,2000-2019,R1,John,1000000,2004


In [90]:
df_customers = pd.DataFrame(customer_records)
df_customers.dtypes

Customer ID          int64
Store Number         int64
Customer Segment    object
Responder           object
Postal Area          int64
dtype: object

In [91]:
df_merge = df_customers.merge(df_postal_codes, on='Postal Area', how='left')
df_merge

,Customer ID,Store Number,Customer Segment,Responder,Postal Area,Range,Region,Sales Rep,Expect Revenue
0,2,100,Corporate,No,2086,2080-2100,R5,Paul,1232345
1,3,100,Corporate,No,2051,2040-2059,R3,Nick,456654
2,5,100,Home Office,No,2077,2060-2079,R4,Mike,234545
3,6,106,Home Office,No,2004,2000-2019,R1,John,1000000
4,8,101,Home Office,No,2010,2000-2019,R1,John,1000000
...,...,...,...,...,...,...,...,...,...
2673,3403,107,Consumer,No,2044,2040-2059,R3,Nick,456654
2674,3390,101,Corporate,No,2053,2040-2059,R3,Nick,456654
2675,3393,106,Consumer,No,2022,2020-2039,R2,Ted,3245234
2676,3391,105,Corporate,No,2093,2080-2100,R5,Paul,1232345


In [92]:
# aggregate the data by Region, Sales Rep, and Responder, get a count of customers
df_agg = df_merge.groupby(['Region', 'Sales Rep', 'Responder'], as_index=False)['Customer ID'].agg('count').rename(columns= {'Customer ID': 'Count'})
df_agg

,Region,Sales Rep,Responder,Count
0,R1,John,No,476
1,R1,John,Yes,76
2,R2,Ted,No,415
3,R2,Ted,Yes,87
4,R3,Nick,No,493
5,R3,Nick,Yes,92
6,R4,Mike,No,430
7,R4,Mike,Yes,82
8,R5,Paul,No,434
9,R5,Paul,Yes,93


In [93]:
print(df_agg.eq(df_output))

   Region  Sales Rep  Responder  Count
0    True       True       True   True
1    True       True       True   True
2    True       True       True   True
3    True       True       True   True
4    True       True       True   True
5    True       True       True   True
6    True       True       True   True
7    True       True       True   True
8    True       True       True   True
9    True       True       True   True
